# 04 — Feature engineering

**Données** : `train.parquet` (80 000 clients), validation croisée 5 folds (`StratifiedKFold`, `random_state` = 42), **mêmes folds** pour toutes les comparaisons.

`churn.features.build.FeatureBuilder` est un transformer sklearn **sans état appris** : chaque feature est une règle fixe (seuil, ratio, indicateur). Il peut donc être placé dans le pipeline sans fuite. Les familles sont activables par paramètre (`features.groups` dans `configs/config.yaml`).

> **Limite à connaître** : les seuils 11-12 mois et 300 jours viennent de l'EDA (E4), faite sur le train entier. Ils ont été choisis en voyant le churn des lignes qui servent ici de folds de validation : la validation croisée est donc **légèrement optimiste** pour la famille `cycle_engagement`. Le jeu de test, lui, n'a jamais été lu : l'évaluation finale (E9) reste impartiale.

In [1]:
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display

from churn import eda
from churn.charts import ablation_chart, publish, rate_chart
from churn.config import get_config
from churn.data.split import load_train
from churn.evaluation import stats as st
from churn.evaluation.cv import ablation, make_folds, summarize_ablation
from churn.features.build import ALL_GROUPS, FEATURE_GROUPS, FeatureBuilder
from churn.logging_setup import setup_logging
from churn.models.pipelines import load_logreg_drop_list, make_lgbm_pipeline, make_logreg_pipeline

setup_logging()
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)

cfg = get_config()
TARGET = cfg.data.target
train = load_train()
y = train[TARGET]
OVERALL = 100 * y.mean()
features = FeatureBuilder(drop_columns=cfg.non_feature_columns).fit_transform(train)
print(train.shape, "->", features.shape)

(80000, 104) -> (80000, 131)


## 1. Les familles et leur justification

| Famille | Feature | Justification métier |
|---|---|---|
| **cycle_engagement** | `in_contract_end` | `months` ∈ {11, 12} : fin supposée de l'engagement de 12 mois, pic de churn en EDA (seuil issu de l'EDA) |
| | `handset_old` | `eqpdays` ≥ 300 : rupture du churn vers 305 jours, terminal proche de l'échéance (seuil issu de l'EDA) |
| | `eqpdays_per_tenure_day` | âge du terminal / ancienneté : un terminal plus vieux que l'abonnement signale un client venu avec son appareil |
| **tendance_usage** | `ratio_mou_3m_6m` | minutes des 3 derniers mois / 6 derniers mois : < 1 = usage en baisse, signe avant-coureur |
| | `ratio_rev_3m_6m` | même ratio sur la facture : baisse de dépense ou changement de forfait |
| **qualite_reseau** | `drop_blk_rate` | part des appels coupés ou bloqués : l'irritant réseau rapporté au volume |
| | `complete_rate` | part des appels aboutis : qualité perçue, indépendante du volume |
| **service_client** | `has_custcare` | au moins un appel au service client (56 % n'appellent jamais) |
| | `custcare_per_mou` | appels au service client par minute d'usage : sollicitation relative |
| **forfait** | `overage_share_rev` | part des dépassements dans la facture : forfait mal adapté |
| | `recurring_share_rev` | part de l'abonnement fixe dans la facture : client « au forfait » |
| **compte_terminal** | `active_lines_share` | lignes actives / lignes ouvertes : lignes résiliées sur le compte |
| | `phones_per_month` | terminaux par mois d'ancienneté : fréquence de renouvellement |
| **indicateurs_manquants** | 15 `manquant_*` + 4 `*_was_negative` | un indicateur par motif de manquants (D22) et par correction de négatifs (D19) : l'absence d'information est elle-même informative |
| **deja_parti** | `sans_usage`, `change_manquant` | aucune donnée d'usage, variation d'usage absente : signal fort mais peut-être **conséquence** du départ (D32), isolé pour le test de fuite en E8 |

### Pic à 23-24 mois ? (condition pour ajouter `months_mod_12`)

In [2]:
tenure = train["months"]
check = pd.DataFrame({
    "groupe": ["11-12 mois", "23-24 mois", "35-36 mois", "13-40 mois hors 23-24 et 35-36"],
    "masque": [tenure.isin([11, 12]), tenure.isin([23, 24]), tenure.isin([35, 36]),
               tenure.between(13, 40) & ~tenure.isin([23, 24, 35, 36])],
})
check["n"] = [int(m.sum()) for m in check["masque"]]
check["taux_churn"] = [round(100 * y[m].mean(), 1) for m in check["masque"]]
check.drop(columns="masque").set_index("groupe")

,n,taux_churn
groupe,,
11-12 mois,11251,63.5
23-24 mois,4759,51.7
35-36 mois,1695,48.4
13-40 mois hors 23-24 et 35-36,44848,51.3


**Conclusion** : pas de second pic (23-24 mois : 51,7 %, contre 51,2 % pour les autres clients de 13 à 40 mois ; 35-36 mois : 48,4 %). Seule la **première** échéance ressort : un client qui a passé le cap des 12 mois ne se comporte plus différemment aux échéances suivantes (réengagement sans nouvelle promotion, ou contrats sans durée). **On ne garde que le flag `in_contract_end`**, pas `months_mod_12`.

## 2. Taux de churn par classe des nouvelles features

Déciles pour les ratios, modalités pour les indicateurs. Même mesure qu'en E5 : écart maximal entre classes (≥ 200 clients), forme de la courbe, information mutuelle.

In [3]:
new_features = [f for g in ALL_GROUPS for f in FEATURE_GROUPS[g]]
data = features.assign(**{TARGET: y})
rows = []
for group in ALL_GROUPS:
    for feat in FEATURE_GROUPS[group]:
        binary = features[feat].nunique() <= 2
        spread, shape = st.class_profile(data, feat, TARGET, numeric=not binary)
        row = {"famille": group, "feature": feat, "type": "indicateur" if binary else "ratio",
               "ecart_classes_pts": round(spread, 1), "forme": shape,
               "info_mutuelle": round(st.mutual_information(data, feat, TARGET, numeric=not binary), 4)}
        if binary:
            row["churn_si_1"] = round(100 * y[features[feat] == 1].mean(), 1)
            row["churn_si_0"] = round(100 * y[features[feat] == 0].mean(), 1)
            row["part_1_pct"] = round(100 * features[feat].mean(), 2)
        rows.append(row)
profile = pd.DataFrame(rows).set_index("feature")
profile

,famille,type,ecart_classes_pts,forme,info_mutuelle,churn_si_1,churn_si_0,part_1_pct
feature,,,,,,,,
in_contract_end,cycle_engagement,indicateur,16.2,sans objet,0.0064,63.5,47.3,14.06
handset_old,cycle_engagement,indicateur,17.0,sans objet,0.0139,56.3,39.4,60.02
eqpdays_per_tenure_day,cycle_engagement,ratio,17.0,non monotone,0.0172,NaN,NaN,NaN
ratio_mou_3m_6m,tendance_usage,ratio,14.3,non monotone,0.0041,NaN,NaN,NaN
ratio_rev_3m_6m,tendance_usage,ratio,4.0,non monotone,0.0016,NaN,NaN,NaN
drop_blk_rate,qualite_reseau,ratio,7.2,non monotone,0.0000,NaN,NaN,NaN
complete_rate,qualite_reseau,ratio,6.9,monotone,0.0031,NaN,NaN,NaN
has_custcare,service_client,indicateur,4.9,sans objet,0.0012,46.8,51.7,44.32
custcare_per_mou,service_client,ratio,5.4,non monotone,0.0000,NaN,NaN,NaN


In [4]:
for feat, title in [
    ("eqpdays_per_tenure_day", "Un terminal plus ancien que l'abonnement va avec plus de churn"),
    ("ratio_mou_3m_6m", "Un usage récent en baisse par rapport aux 6 mois va avec plus de churn"),
    ("overage_share_rev", "La part des dépassements dans la facture a un effet modeste"),
    ("phones_per_month", "Changer souvent de terminal va avec moins de churn"),
]:
    publish(rate_chart(eda.churn_rate_table(data, eda.decile_bins(features[feat])), title, OVERALL,
                       f"décile de {feat}"), f"04_rate_{feat}")

**Lecture des nouvelles features** (associations sur le train) :

- **cycle_engagement** : `in_contract_end` isole 14,1 % des clients à **63,5 %** de churn (47,3 % sinon) ; `handset_old` 56,3 % contre 39,4 % ; `eqpdays_per_tenure_day` a la plus forte information mutuelle des nouvelles features (0,017, au niveau de `eqpdays`) ;
- **compte_terminal** : `phones_per_month` a un écart de 19,9 points (courbe non monotone) : elle contient surtout `months` au dénominateur et hérite de son pic ;
- **tendance_usage** : `ratio_mou_3m_6m` (écart 14,3 pts) : un usage récent en baisse va avec plus de churn ;
- **forfait** : `recurring_share_rev` (11,4 pts) et `overage_share_rev` (8,4 pts), effets monotones modestes ;
- **qualite_reseau** et **service_client** : information mutuelle quasi nulle (≤ 0,003), écarts de 5 à 7 points ;
- **indicateurs_manquants** : seuls `manquant_hnd_webcap` (+10,4 pts), `manquant_hnd_price` (−12,1 pts) et `manquant_avg6mou_bloc` (−9,8 pts) se distinguent ; les indicateurs de négatifs concernent chacun moins de 0,2 % des clients ;
- **deja_parti** : `change_manquant` 76,0 % et `sans_usage` 68,9 % de churn, sur moins de 1 % des clients.

## 3. Ablation par famille (validation croisée 5 folds, mêmes folds)

Deux modèles :

- **LightGBM** aux hyperparamètres par défaut (gère NaN et catégorielles nativement) ;
- **régression logistique** simple : imputation médiane + standardisation + one-hot, **liste de retrait de E5 appliquée** (36 variables), le tout dans un `Pipeline` ajusté dans chaque fold.

Configurations : **référence** (variables d'origine seules), référence + **chaque famille**, puis **toutes les familles**. Le gain est calculé fold par fold contre la référence.

In [5]:
folds = make_folds(y)
drop_list = load_logreg_drop_list()
configs = {"référence": []} | {f"+ {g}": [g] for g in ALL_GROUPS} | {"toutes les familles": list(ALL_GROUPS)}
factories = {
    "LightGBM": lambda groups: make_lgbm_pipeline(groups),
    "Régression logistique": lambda groups: make_logreg_pipeline(groups, drop_list),
}
raw_results = ablation(configs, factories, train, y, folds)
summary = summarize_ablation(raw_results, reference="référence")
print(f"{len(raw_results)} ajustements ({len(configs)} configurations × {len(factories)} modèles × {len(folds)} folds)")

12:10:41 | INFO    | churn.evaluation.cv | LightGBM | référence : AUC 0.6899 ± 0.0047


12:10:46 | INFO    | churn.evaluation.cv | LightGBM | + cycle_engagement : AUC 0.6904 ± 0.0044


12:10:51 | INFO    | churn.evaluation.cv | LightGBM | + tendance_usage : AUC 0.6902 ± 0.0043


12:10:56 | INFO    | churn.evaluation.cv | LightGBM | + qualite_reseau : AUC 0.6900 ± 0.0047


12:11:01 | INFO    | churn.evaluation.cv | LightGBM | + service_client : AUC 0.6900 ± 0.0047


...

12:13:14 | INFO    | churn.evaluation.cv | Régression logistique | + compte_terminal : AUC 0.6242 ± 0.0033


12:13:30 | INFO    | churn.evaluation.cv | Régression logistique | + indicateurs_manquants : AUC 0.6233 ± 0.0037


12:13:46 | INFO    | churn.evaluation.cv | Régression logistique | + deja_parti : AUC 0.6247 ± 0.0033


12:14:03 | INFO    | churn.evaluation.cv | Régression logistique | toutes les familles : AUC 0.6574 ± 0.0031


100 ajustements (10 configurations × 2 modèles × 5 folds)


In [6]:
table = summary.pivot_table(index="configuration", columns="modele",
                            values=["auc_moyenne", "auc_ecart_type", "gain_moyen", "gain_ecart_type"])
out = pd.DataFrame(index=list(configs))
for model in factories:
    out[f"{model} AUC"] = [f"{table.loc[c, ('auc_moyenne', model)]:.4f} ± {table.loc[c, ('auc_ecart_type', model)]:.4f}" for c in configs]
    out[f"{model} gain"] = [f"{1000 * table.loc[c, ('gain_moyen', model)]:+.1f} ± {1000 * table.loc[c, ('gain_ecart_type', model)]:.1f}" for c in configs]
out.index.name = "configuration (gain en millièmes d'AUC, ± écart-type par fold)"
out

,LightGBM AUC,LightGBM gain,Régression logistique AUC,Régression logistique gain
"configuration (gain en millièmes d'AUC, ± écart-type par fold)",,,,
référence,0.6899 ± 0.0053,+0.0 ± 0.0,0.6225 ± 0.0040,+0.0 ± 0.0
+ cycle_engagement,0.6904 ± 0.0049,+0.5 ± 1.3,0.6508 ± 0.0045,+28.2 ± 2.7
+ tendance_usage,0.6902 ± 0.0048,+0.3 ± 1.9,0.6255 ± 0.0027,+2.9 ± 1.5
+ qualite_reseau,0.6900 ± 0.0053,+0.1 ± 0.7,0.6245 ± 0.0044,+1.9 ± 1.2
+ service_client,0.6900 ± 0.0052,+0.1 ± 1.0,0.6224 ± 0.0038,-0.1 ± 0.5
+ forfait,0.6909 ± 0.0051,+0.9 ± 0.9,0.6233 ± 0.0041,+0.8 ± 0.3
+ compte_terminal,0.6913 ± 0.0047,+1.3 ± 1.6,0.6242 ± 0.0037,+1.6 ± 0.5
+ indicateurs_manquants,0.6903 ± 0.0045,+0.4 ± 1.4,0.6233 ± 0.0042,+0.7 ± 0.3
+ deja_parti,0.6899 ± 0.0053,+0.0 ± 0.0,0.6247 ± 0.0037,+2.1 ± 0.6


In [7]:
publish(ablation_chart(summary, "référence",
                      "Le cycle d'engagement transforme la régression logistique ; LightGBM n'a besoin d'aucune famille",
                      order=list(configs)), "04_ablation", width=1000)
summary.to_csv(cfg.paths.reports_dir / "ablation_features.csv", index=False, encoding="utf-8")

**Lecture** :

- **LightGBM** (AUC de référence 0,690 ± 0,005) : **aucune famille ne dépasse l'écart-type entre folds** ; le meilleur apport est de +1,3 millième (`compte_terminal`), toutes familles réunies +1,3 ± 2,5. Les arbres trouvent eux-mêmes les seuils (11-12 mois, 300 jours) et les ratios utiles dans les variables brutes.
- **Régression logistique** (0,623 ± 0,004) : **`cycle_engagement` apporte +28,2 millièmes, dans 5 folds sur 5**. Un modèle linéaire ne peut pas représenter un pic à 11-12 mois ni une marche à 300 jours : il faut le lui donner. Les autres familles apportent de +0,7 à +2,9 millièmes, régulièrement (5 folds sur 5 pour 5 familles) mais sous l'écart-type de 4 millièmes.
- **`deja_parti` : exactement 0,0 pour LightGBM**, dans aucun fold. LightGBM exploite **déjà** les NaN de `rev_Mean` et `change_mou`, qui portent la même information. Conséquence pour le test de fuite D32 (E8) : désactiver la famille ne suffit pas pour LightGBM ; il faudra aussi neutraliser les NaN d'origine (imputation, ou retrait des clients concernés).
- L'écart entre les deux modèles reste grand (0,690 contre 0,657 avec toutes les familles) : même aidée, la régression logistique ne capte pas toutes les interactions.

## 4. Décision par famille

Règle : **garder** si le gain moyen dépasse l'écart-type de l'AUC de référence entre folds pour au moins un des deux modèles ; sinon, garder seulement si la famille est neutre mais utile à l'interprétation (justification écrite).

In [8]:
ref = summary[summary["configuration"] == "référence"].set_index("modele")["auc_ecart_type"]
decisions = []
for group in ALL_GROUPS:
    row = {"famille": group}
    for model in factories:
        s = summary[(summary["configuration"] == f"+ {group}") & (summary["modele"] == model)].iloc[0]
        row[f"{model} gain (millièmes)"] = round(1000 * s["gain_moyen"], 1)
        row[f"{model} gain > écart-type"] = bool(s["gain_moyen"] > ref[model])
        row[f"{model} folds en hausse"] = f"{s['folds_en_hausse']}/5"
    row["critère rempli"] = any(row[f"{m} gain > écart-type"] for m in factories)
    decisions.append(row)
decisions = pd.DataFrame(decisions).set_index("famille")
print("écart-type de l'AUC de référence :", {m: round(v, 4) for m, v in ref.items()})
decisions

écart-type de l'AUC de référence : {'LightGBM': 0.0053, 'Régression logistique': 0.004}


,LightGBM gain (millièmes),LightGBM gain > écart-type,LightGBM folds en hausse,Régression logistique gain (millièmes),Régression logistique gain > écart-type,Régression logistique folds en hausse,critère rempli
famille,,,,,,,
cycle_engagement,0.5,False,3/5,28.2,True,5/5,True
tendance_usage,0.3,False,2/5,2.9,False,5/5,False
qualite_reseau,0.1,False,3/5,1.9,False,4/5,False
service_client,0.1,False,2/5,-0.1,False,3/5,False
forfait,0.9,False,4/5,0.8,False,5/5,False
compte_terminal,1.3,False,4/5,1.6,False,5/5,False
indicateurs_manquants,0.4,False,3/5,0.7,False,5/5,False
deja_parti,0.0,False,0/5,2.1,False,5/5,False


**Décisions** :

Seule `cycle_engagement` remplit le critère strict. Pour les familles neutres, la règle retenue : garder si le gain est positif dans **5 folds sur 5** pour au moins un modèle **et** si la famille sert un levier d'action ou l'explication d'un score.

| Famille | Décision | Justification |
|---|---|---|
| cycle_engagement | **Garder** | Critère rempli : +28,2 millièmes pour la régression logistique, 5/5 folds |
| tendance_usage | **Garder** (neutre, utile) | +2,9 ± 1,5, 5/5 folds (régression logistique) ; explique un score par « usage en baisse », levier : relance d'usage |
| forfait | **Garder** (neutre, utile) | +0,8 ± 0,3, 5/5 folds ; « dépassements élevés » appelle un changement de forfait |
| compte_terminal | **Garder** (neutre, utile) | +1,6 ± 0,5 (5/5) pour la régression logistique, meilleure famille pour LightGBM (+1,3, 4/5) ; levier : renouvellement du terminal |
| indicateurs_manquants | **Garder** (neutre, utile) | +0,7 ± 0,3, 5/5 ; indispensable à la régression logistique, dont l'imputation médiane efface sinon l'information « donnée absente » (D22) |
| qualite_reseau | **Retirer** | 4/5 folds seulement, information mutuelle quasi nulle ; les variables brutes de qualité réseau restent disponibles |
| service_client | **Retirer** | Effet nul (−0,1 et +0,1) ; `custcare_Mean` brut reste disponible |
| deja_parti | **Hors configuration par défaut** | Suspect de fuite (D32) ; conservé dans le code pour le test en E8 |

### Vérification : configuration retenue, mêmes folds

In [9]:
KEPT = ["cycle_engagement", "tendance_usage", "forfait", "compte_terminal", "indicateurs_manquants"]
final_configs = {"référence": [], "familles retenues": KEPT, "toutes les familles": list(ALL_GROUPS)}
final_results = ablation({"familles retenues": KEPT}, factories, train, y, folds)
final = summarize_ablation(pd.concat([raw_results, final_results]), reference="référence")
final = final[final["configuration"].isin(final_configs)]
final.assign(**{c: final[c].round(4) for c in ["auc_moyenne", "auc_ecart_type", "gain_moyen", "gain_ecart_type"]})      .set_index(["modele", "configuration"]).loc[:, ["auc_moyenne", "auc_ecart_type", "gain_moyen", "gain_ecart_type", "folds_en_hausse"]]

12:14:11 | INFO    | churn.evaluation.cv | LightGBM | familles retenues : AUC 0.6917 ± 0.0038


12:14:26 | INFO    | churn.evaluation.cv | Régression logistique | familles retenues : AUC 0.6542 ± 0.0034


auc_moyenne  auc_ecart_type  gain_moyen  gain_ecart_type  folds_en_hausse
modele                configuration                                                                                 
LightGBM              familles retenues         0.6917          0.0043      0.0018           0.0026                3
                      référence                 0.6899          0.0053      0.0000           0.0000                0
                      toutes les familles       0.6912          0.0048      0.0013           0.0025                3
Régression logistique familles retenues         0.6542          0.0038      0.0316           0.0028                5
                      référence                 0.6225          0.0040      0.0000           0.0000                0
                      toutes les familles       0.6574          0.0034      0.0349           0.0026                5

## Conclusion

- **Configuration retenue** (`features.groups` dans `configs/config.yaml`) : `cycle_engagement`, `tendance_usage`, `forfait`, `compte_terminal`, `indicateurs_manquants` ; 28 features ajoutées.
- **Régression logistique : 0,654 ± 0,004** (référence 0,623, **+31,6 millièmes, 5/5 folds**). **LightGBM : 0,692 ± 0,004** (référence 0,690, +1,8 ± 2,6, dans le bruit).
- Le feature engineering sert donc surtout à **rendre le modèle interprétable compétitif** (la régression logistique comble 43 % de son retard sur LightGBM : l'écart passe de 67 à 38 millièmes) et à fournir des **raisons lisibles** pour l'explication des scores (E11) ; il ne change pas la performance d'un modèle à arbres.
- Point à reprendre en E8 : le test de fuite `deja_parti` doit aussi neutraliser les NaN d'origine pour LightGBM.
- Limite : les seuils de `cycle_engagement` (11-12 mois, 300 jours) ont été fixés en EDA sur le train entier ; le gain de +28,2 millièmes en CV est donc légèrement optimiste. Le test final (E9) tranchera.